# DDPM Training On Kaggle

This notebook runs the project end-to-end on Kaggle GPU.

Required accelerator for this notebook: **T4**.

If Kaggle assigns **P100**, stop and switch the notebook accelerator to T4 from Notebook Settings before continuing.

In [ ]:
import os

# TODO: replace with your GitHub repository URL
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(REPO_DIR):
    print(f"Updating existing repo at {REPO_DIR}")
    !git -C "$REPO_DIR" pull
else:
    print(f"Cloning repo to {REPO_DIR}")
    !git clone "$REPO_URL" "$REPO_DIR"

%cd "$REPO_DIR"
!git log --oneline -n 3

In [ ]:
# Install project dependencies without replacing Kaggle's preinstalled torch build
!pip install -r requirements-kaggle.txt

In [ ]:
# Optional: only run this if you are using P100 and get CUDA architecture errors
# !pip uninstall -y torch torchvision torchaudio
# !pip install --index-url https://download.pytorch.org/whl/cu118 torch==2.3.1 torchvision==0.18.1

In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Kaggle, open Notebook Settings and set Accelerator to GPU (T4)."
    )

device_count = torch.cuda.device_count()
device_names = [torch.cuda.get_device_name(i) for i in range(device_count)]

print("CUDA device count:", device_count)
for i, name in enumerate(device_names):
    print(i, name)

if not any("T4" in name for name in device_names):
    raise RuntimeError(
        "This run is configured for T4 GPUs, but no T4 device was assigned. "
        "Open Notebook Settings, switch Accelerator to GPU, and select T4, then restart session."
    )

if device_count < 2:
    print(
        "Warning: only one T4 detected. Training will still run, but more slowly than 2x T4."
    )
else:
    print("T4 check passed.")

In [ ]:
# Train
!python train.py --config configs/kaggle.yaml

In [ ]:
import glob
import os

run_dirs = sorted(glob.glob("/kaggle/working/artifacts/ddpm_mnist_kaggle/*"))
if not run_dirs:
    raise RuntimeError("No run directories found. Check training output.")

latest = run_dirs[-1]
checkpoint = os.path.join(latest, "checkpoints", "final.pt")
if not os.path.exists(checkpoint):
    raise RuntimeError(f"Checkpoint not found: {checkpoint}")

print("Using checkpoint:", checkpoint)
!python eval.py --config configs/kaggle.yaml --checkpoint "$checkpoint" --num-samples 10000

In [ ]:
# Package artifacts for easy download from Kaggle output panel
!zip -r /kaggle/working/ddpm_artifacts.zip /kaggle/working/artifacts
print("Saved: /kaggle/working/ddpm_artifacts.zip")